# Data Quality and Exploratory Analysis — Customer Churn

**Objective:** Audit and clean the supplied synthetic customer churn data, then document quality decisions and initial patterns.

**Decision question:** Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

Run the notebook from top to bottom. Keep the original CSV in the same folder as this notebook.

## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("customer_churn_sample.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)

## 2. Profile missing values, duplicates, and data types

In [ ]:
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())

print("Missing values by column:")
display(df.isna().sum().rename("missing_count").to_frame())

print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())

for col in df.select_dtypes(include="object").columns:
    print(f"\\n{col}: {sorted(df[col].dropna().unique().tolist())}")

## 3. Clean with documented assumptions

In [ ]:
clean = df.copy()
clean.columns = [c.strip() for c in clean.columns]
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype("string").str.strip()

numeric_cols = ["Age", "TenureMonths", "MonthlyCharges", "TotalCharges", "SupportTickets"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Conservative policy: do not delete rows automatically. Investigate any invalid values first.
quality_checks = {
    "missing_cells": int(clean.isna().sum().sum()),
    "duplicate_rows": int(clean.duplicated().sum()),
    "duplicate_customer_ids": int(clean["CustomerID"].duplicated().sum()),
    "invalid_age": int(((clean["Age"] < 0) | (clean["Age"] > 120)).sum()),
    "negative_tenure": int((clean["TenureMonths"] < 0).sum()),
    "negative_monthly_charges": int((clean["MonthlyCharges"] < 0).sum()),
    "negative_total_charges": int((clean["TotalCharges"] < 0).sum()),
    "negative_support_tickets": int((clean["SupportTickets"] < 0).sum()),
    "invalid_churn_labels": int((~clean["Churn"].isin(["Yes", "No"])).sum()),
}
quality_checks

## 4. Detect possible outliers using the IQR rule

In [ ]:
outlier_rows = []
for col in numeric_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = clean[col].lt(lower) | clean[col].gt(upper)
    outlier_rows.append({
        "column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "flagged_count": int(flagged.sum()),
        "flagged_values": clean.loc[flagged, col].tolist()
    })
outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table)
# IQR flags are review prompts; do not remove values without a domain reason.

## 5. Validate business rules and save cleaned data

In [ ]:
# Check whether total charges match the apparent sample formula within a small rounding tolerance.
charge_diff = (clean["TotalCharges"] - clean["TenureMonths"] * clean["MonthlyCharges"]).abs()
print("TotalCharges formula mismatches (> INR 0.02):", int((charge_diff > 0.02).sum()))

allowed_contracts = {"Month-to-Month", "One Year", "Two Year"}
print("Unexpected contract labels:", sorted(set(clean["ContractType"].dropna()) - allowed_contracts))
print("Unexpected churn labels:", sorted(set(clean["Churn"].dropna()) - {"Yes", "No"}))

clean.to_csv("customer_churn_cleaned.csv", index=False)
print("Saved customer_churn_cleaned.csv")

## 6. Summary statistics

In [ ]:
print("Customer count:", len(clean))
print("Churned customers:", int((clean["Churn"] == "Yes").sum()))
print("Churn rate (%):", round((clean["Churn"] == "Yes").mean() * 100, 1))
display(clean[["Age", "TenureMonths", "MonthlyCharges", "TotalCharges", "SupportTickets"]].describe().T)

## 7. Explore churn patterns

In [ ]:
def churn_summary(column):
    out = clean.groupby(column).agg(
        customers=("CustomerID", "count"),
        churned=("Churn", lambda s: (s == "Yes").sum())
    )
    out["churn_rate_pct"] = (out["churned"] / out["customers"] * 100).round(1)
    return out.sort_values("churn_rate_pct", ascending=False)

display(churn_summary("ContractType"))
display(churn_summary("SubscriptionType"))
display(churn_summary("PaymentMethod"))
display(pd.crosstab(clean["SupportTickets"], clean["Churn"], margins=True))

In [ ]:
clean["Churn"].value_counts().reindex(["No", "Yes"]).plot(kind="bar")
plt.title("Customer Count by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

(churn_summary("ContractType")["churn_rate_pct"]).plot(kind="bar")
plt.title("Observed Churn Rate by Contract Type")
plt.xlabel("Contract type")
plt.ylabel("Churn rate (%)")
plt.ylim(0, 110)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Findings, limitations, and recommendation

- Prioritize reviewing Month-to-Month customers; in this sample, all 7 churned.
- Basic subscription customers had 5 churns among 7 customers (71.4%).
- The overall observed churn rate is 7/15 (46.7%).
- These are descriptive patterns from only 15 synthetic rows, not causal evidence.

**Recommendation:** Pilot a proactive check-in for Month-to-Month customers with multiple support tickets. Track subsequent churn, support-ticket volume, and intervention cost, and compare results against a suitable control or matched group.